# Phase 2 — CatBoost upgrade

After our Phase 1 XGBoost submission, we read the official PHEMS hackathon **Lessons-Learned report** [PHEMS Consortium, Zenodo 17045913, 2025]. The organizers note that the highest-scoring teams used **CatBoost with `auto_class_weights='Balanced'`** instead of XGBoost, LightGBM, or transformer-based models.

**This notebook tests that finding on our own setup:**
- Same feature pipeline as Phase 1 (Khang's `data_prep.py` + `prepare_for_xgboost`).
- Same person-aware train / val / test split.
- Replace XGBoost with CatBoost using the report's recommended class-weighting setting.
- Evaluate and generate a Kaggle submission.

We did not change anything else. If CatBoost outperforms XGBoost on the same features, that's evidence the report's recommendation generalizes.

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier

# Reuse Khang's pipeline functions — feature build, split, scoring
import data_prep
from missing_values import prepare_for_xgboost
from XGBoost.train_XGBoost import load_training_data, split_data_person_aware, score_model

ARTIFACTS = Path('artifacts')
ROOT = Path('phems-hackathon-early-sepsis-prediction')
RANDOM_SEED = 42
print('Setup OK')

Setup OK


## 1. Load features and apply person-aware split
Same as Phase 1 — same data, same split, same patients in each partition.

In [2]:
df, features, labels = load_training_data()
X_train, X_val, X_test, y_train, y_val, y_test = split_data_person_aware(
    df, features, labels, test_size=0.2, val_size=0.2, random_state=RANDOM_SEED
)
print(f'Train: {X_train.shape}, positives: {int(y_train.sum())} ({y_train.mean():.4%})')
print(f'Val:   {X_val.shape}, positives: {int(y_val.sum())} ({y_val.mean():.4%})')
print(f'Test:  {X_test.shape}, positives: {int(y_test.sum())} ({y_test.mean():.4%})')

Train: (198331, 258), positives: 4736 (2.3879%)
Val:   (69344, 258), positives: 1164 (1.6786%)
Test:  (63978, 258), positives: 974 (1.5224%)


## 2. Train CatBoost
Settings come from the organizers' Lessons-Learned report:
- `auto_class_weights='Balanced'` — automatic class-weight balancing (the report's key recommendation).
- `eval_metric='PRAUC'` — track the competition metric for early stopping.
- Modest depth/learning rate; rely on early stopping for the right number of iterations.

In [3]:
model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    auto_class_weights='Balanced',
    eval_metric='PRAUC',
    loss_function='Logloss',
    early_stopping_rounds=30,
    random_seed=RANDOM_SEED,
    verbose=False,
)
model.fit(
    X_train.to_numpy(), y_train,
    eval_set=(X_val.to_numpy(), y_val),
    verbose=False,
)
print(f'Best iteration (early-stopped): {model.get_best_iteration()}')

Best iteration (early-stopped): 64


## 3. Evaluate on each split
Reuses Khang's `score_model()` for an apples-to-apples comparison with Phase 1.

In [4]:
print('=' * 60)
print('CATBOOST — performance on each split')
print('=' * 60)
score_model(model, X_train, y_train, 'Train')
print()
score_model(model, X_val,   y_val,   'Validation')
print()
score_model(model, X_test,  y_test,  'Test')

CATBOOST — performance on each split


Train ROC AUC:   0.9847
Train PR AUC:    0.8102
Train Accuracy:  0.8656 @ threshold=0.20
Train Recall:    0.9753 @ threshold=0.20
Train Precision: 0.1482 @ threshold=0.20



Validation ROC AUC:   0.9537
Validation PR AUC:    0.4000
Validation Accuracy:  0.8731 @ threshold=0.20
Validation Recall:    0.9313 @ threshold=0.20
Validation Precision: 0.1106 @ threshold=0.20



Test ROC AUC:   0.9603
Test PR AUC:    0.2920
Test Accuracy:  0.8683 @ threshold=0.20
Test Recall:    0.9517 @ threshold=0.20
Test Precision: 0.0996 @ threshold=0.20


## 4. Generate Kaggle submission
Same submission format as Phase 1: `person_id_datetime,SepsisLabel`, 130,483 rows.

In [5]:
test_df = pd.read_csv(ARTIFACTS / 'test_features.csv')
ids = test_df['person_id'].astype(str) + '_' + test_df['measurement_datetime'].astype(str)

X_kaggle = test_df.drop(columns=['person_id', 'measurement_datetime'], errors='ignore')
X_kaggle = prepare_for_xgboost(X_kaggle, label_column=None, add_indicators=True, add_summary=True)
X_kaggle = X_kaggle.reindex(columns=features.columns)
print(f'Kaggle test features: {X_kaggle.shape}')

proba = model.predict_proba(X_kaggle.to_numpy())[:, 1]
print(f'Predictions: mean={proba.mean():.4f}, p99={np.quantile(proba, 0.99):.4f}')

sample = pd.read_csv(ROOT / 'SepsisLabel_sample_submission.csv')
sub = pd.DataFrame({'person_id_datetime': ids, 'SepsisLabel': proba})
assert sub.shape[0] == sample.shape[0]
assert set(sub['person_id_datetime']) == set(sample['person_id_datetime'])
sub = sub.set_index('person_id_datetime').reindex(sample['person_id_datetime']).reset_index()

out_path = ARTIFACTS / 'submission_catboost.csv'
sub.to_csv(out_path, index=False)
print(f'Wrote {out_path} ({len(sub):,} rows)')
print(sub.head())

Kaggle test features: (130483, 258)


Predictions: mean=0.1245, p99=0.9593
Wrote artifacts/submission_catboost.csv (130,483 rows)
               person_id_datetime  SepsisLabel
0  1416048048_2021-03-25 10:00:00     0.364058
1   280531880_2024-01-22 18:00:00     0.066437
2  1127023302_2023-12-29 21:00:00     0.039148
3  2065909112_2021-07-07 05:00:00     0.046601
4   264445818_2024-08-23 22:00:00     0.028639
